### RAG Pipeline - Data Ingestion to Vector DB Pipeline 


In [ ]:
import os
from pathlib import Path
from bs4 import BeautifulSoup
from langchain_core.documents import Document
import time


In [ ]:
### Read all the HTML files inside the directory

from langchain_core.documents import Document

def load_html_file(html_file):
    """Parse a Unity docs HTML file into a LangChain Document."""
    with html_file.open("r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    h1 = soup.find("h1")
    article = h1.find_parent("div", class_="section") if h1 else None
    if article is None:
        article = soup.find("body") or soup

    for noise in article.select("div.breadcrumbs, div.nextprev, script, style, span.tooltiptext"):
        noise.decompose()

    title = h1.get_text(" ", strip=True) if h1 else html_file.stem
    text = article.get_text("\n", strip=True)

    if not text.strip():
        raise ValueError("No article text extracted")

    return Document(
        page_content=text,
        metadata={
            "source": str(html_file),
            "source_file": html_file.name,
            "file_type": "html",
            "title": title,
        },
    )


def process_all_data(data_directory="../data"):
    """Process all HTML files in a directory."""
    all_documents = []
    data_dir = Path(data_directory)

    html_files = sorted(data_dir.glob("**/*.html"))

    print(f"Found {len(html_files)} HTML files to process")

    for html_file in html_files:
        print(f"\nProcessing: {html_file.name}")
        try:
            document = load_html_file(html_file)
            all_documents.append(document)
            print("  ✓ Loaded 1 document")
        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


all_html_documents = process_all_data("../data")


In [ ]:
### Text Splitting - get into Chunks

from pathlib import Path
from bs4 import BeautifulSoup
from langchain_core.documents import Document


def parse_html_article(html_file):
    """Load an HTML file and return the cleaned article node plus title."""
    html_file = Path(html_file)
    with html_file.open("r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    h1 = soup.find("h1")
    article = h1.find_parent("div", class_="section") if h1 else None
    if article is None:
        article = soup.find("body") or soup

    for noise in article.select("div.breadcrumbs, div.nextprev, script, style, span.tooltiptext"):
        noise.decompose()

    title = h1.get_text(" ", strip=True) if h1 else html_file.stem
    return article, title


def extract_blocks(article):
    """Split an HTML article into semantic blocks (headings, paragraphs, lists, code, tables)."""
    blocks = []
    current_section = None
    seen_h1 = False

    elements = article.find_all([
        "h1",
        "h2",
        "h3",
        "h4",
        "p",
        "li",
        "pre",
        "table",
    ])

    for element in elements:
        if element.name in ["h1", "h2", "h3", "h4"]:
            text = element.get_text(" ", strip=True)
            if not text:
                continue

            if element.name == "h1":
                seen_h1 = True

            current_section = text
            blocks.append({
                "type": "heading",
                "tag": element.name,
                "section": current_section,
                "text": text,
            })

        elif not seen_h1:
            continue

        elif element.name == "p":
            text = element.get_text(" ", strip=True)
            if not text:
                continue
            blocks.append({
                "type": "paragraph",
                "tag": "p",
                "section": current_section,
                "text": text,
            })

        elif element.name == "li":
            text = element.get_text(" ", strip=True)
            if not text:
                continue
            blocks.append({
                "type": "list_item",
                "tag": "li",
                "section": current_section,
                "text": text,
            })

        elif element.name == "pre":
            text = element.get_text("\n", strip=True)
            if not text:
                continue
            blocks.append({
                "type": "code",
                "tag": "pre",
                "section": current_section,
                "text": text,
            })

        elif element.name == "table":
            rows = []
            for tr in element.find_all("tr"):
                cells = tr.find_all(["th", "td"])
                row = [cell.get_text(" ", strip=True) for cell in cells]
                if row:
                    rows.append(row)

            if not rows:
                continue

            headers = rows[0]
            for row in rows[1:]:
                if len(row) != len(headers):
                    continue
                table_text = "\n".join(
                    f"{header}: {value}"
                    for header, value in zip(headers, row)
                )
                blocks.append({
                    "type": "table_row",
                    "tag": "table",
                    "section": current_section,
                    "text": table_text,
                })

    return blocks


def split_documents(documents):
    """Chunk loaded HTML documents using structured article blocks."""
    all_chunks = []

    print(f"Splitting {len(documents)} documents into semantic chunks")

    for doc in documents:
        html_file = Path(doc.metadata["source"])
        print(f"\nChunking: {html_file.name}")
        try:
            article, title = parse_html_article(html_file)
            blocks = extract_blocks(article)

            for i, block in enumerate(blocks):
                all_chunks.append(
                    Document(
                        page_content=block["text"],
                        metadata={
                            **doc.metadata,
                            "title": title,
                            "block_type": block["type"],
                            "tag": block["tag"],
                            "section": block["section"],
                            "block_index": i,
                        },
                    )
                )

            print(f"  ✓ Split into {len(blocks)} chunks")
        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal chunks: {len(all_chunks)}")
    return all_chunks


all_chunks = split_documents(all_html_documents)

#all_chunks


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List


In [ ]:
### Generate embeddings

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L12-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            dim_fn = getattr(self.model, "get_embedding_dimension", None) or getattr(
                self.model, "get_sentence_embedding_dimension"
            )
            print(f"Model loaded successfully. Embedding dimension: {dim_fn()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


#initialize embedding manager  
 
embedding_manager = EmbeddingManager()
embedding_manager


### VectorStore


In [ ]:
import os
from typing import List

import chromadb
import numpy as np
from langchain_core.documents import Document


class VectorStore:
    """Stores document chunks and embeddings in a persistent ChromaDB collection."""

    def __init__(
        self,
        embedding_manager: EmbeddingManager,
        store_path: str = "../data/vector_store",
        collection_name: str = "html_documents",
    ):
        """
        Initialize the vector store

        Args:
            embedding_manager: Embedding manager instance
            store_path: Path to store the vector database
            collection_name: ChromaDB collection name
        """
        self.embedding_manager = embedding_manager
        self.store_path = store_path
        self.collection_name = collection_name
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Create or open a persistent ChromaDB database and collection."""
        try:
            os.makedirs(self.store_path, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.store_path)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"},
            )
            print(f"Vector store initialized at {self.store_path}")
            print(f"Collection: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def _clean_metadata(self, metadata: dict) -> dict:
        """ChromaDB metadata values must be str, int, float, or bool."""
        cleaned = {}
        for key, value in metadata.items():
            if value is None:
                cleaned[key] = ""
            elif isinstance(value, (str, int, float, bool)):
                cleaned[key] = value
            else:
                cleaned[key] = str(value)
        return cleaned

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add document chunks and their embeddings to ChromaDB.

        Args:
            documents: Chunked Document objects
            embeddings: Embedding matrix with shape (len(documents), embedding_dim)
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        if self.collection.count() > 0:
            print(
                f"Resetting collection ({self.collection.count()} existing docs) "
                "to avoid duplicates on re-run"
            )
            self.client.delete_collection(self.collection_name)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"},
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            source_file = doc.metadata.get("source_file", "document")
            block_index = doc.metadata.get("block_index", i)
            doc_id = f"{source_file}::{block_index}"
            ids.append(doc_id)

            metadata = self._clean_metadata(dict(doc.metadata))
            metadata["doc_index"] = i
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(
                embedding.tolist() if hasattr(embedding, "tolist") else list(embedding)
            )

        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            documents=documents_text,
            metadatas=metadatas,
        )

        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in store: {self.collection.count()}")

    def query(self, query_text: str, n_results: int = 5):
        """Embed a question and return the most similar stored chunks."""
        query_embedding = self.embedding_manager.generate_embeddings([query_text])
        return self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )


vector_store = VectorStore(embedding_manager)
vector_store


In [ ]:
### Convert the HTML chunks to embeddings
texts = [doc.page_content for doc in all_chunks]

## Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vector_store.add_documents(all_chunks, embeddings)


### Retriever Pipeline from VectorStore


In [ ]:
from typing import Any, Dict, List


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1,
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [ ]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever


### VectorDB to Output Generation 


In [ ]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
load_dotenv(Path("../.env"))


class RAGGenerator:
    """Retrieve Unity HTML chunks from ChromaDB and generate an answer."""

    def __init__(
        self,
        retriever: RAGRetriever,
        model_name: str = "gemini-3.5-flash-lite",
    ):
        """
        Args:
            retriever: RAGRetriever backed by the HTML vector store
            model_name: Gemini model. gemini-3.5-flash-lite is on the free tier.
        """
        if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
            raise ValueError(
                "GOOGLE_API_KEY or GEMINI_API_KEY is not set. Add it to a .env file in the project root."
            )

        self.retriever = retriever
        self.model_name = model_name
        self.llm = ChatGoogleGenerativeAI(model=model_name)
        print(f"Loaded Gemini model: {model_name}")

    def _page_url(self, metadata: dict) -> str:
        """Resolve the original Unity HTML page URL for a retrieved chunk."""
        source = metadata.get("source")
        source_path = Path(source) if source else None
        if source_path and source_path.exists():
            html = source_path.read_text(encoding="utf-8", errors="replace")
            match = re.search(r'rel="canonical" href="([^"]+)"', html)
            if match:
                return match.group(1)

        stem = Path(metadata.get("source_file", "")).stem
        if stem:
            return f"https://docs.unity3d.com/6000.7/Documentation/Manual/{stem}.html"
        return ""

    def build_citations(self, retrieved_docs: list) -> list:
        """Build numbered citations pointing at the original HTML pages."""
        citations = []
        for i, doc in enumerate(retrieved_docs, start=1):
            metadata = doc.get("metadata") or {}
            citations.append({
                "id": i,
                "title": metadata.get("title", ""),
                "section": metadata.get("section", ""),
                "source_file": metadata.get("source_file", "unknown"),
                "url": self._page_url(metadata),
                "score": doc.get("similarity_score", 0.0),
            })
        return citations

    def _format_context(self, retrieved_docs: list) -> str:
        """Join retrieved HTML chunks into a numbered citation context string."""
        if not retrieved_docs:
            return ""

        parts = []
        citations = self.build_citations(retrieved_docs)
        for doc, citation in zip(retrieved_docs, citations):
            header = (
                f"[{citation['id']}] {citation['title'] or citation['source_file']}"
            )
            if citation["section"]:
                header += f" | Section: {citation['section']}"
            if citation["url"]:
                header += f" | {citation['url']}"
            parts.append(f"{header}\n{doc['content']}")
        return "\n\n".join(parts)

    def _as_text(self, content) -> str:
        """Normalize LangChain/Gemini message content to a plain string."""
        if content is None:
            return ""
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, dict) and item.get("text"):
                    parts.append(item["text"])
                elif hasattr(item, "text"):
                    parts.append(item.text)
                else:
                    parts.append(str(item))
            return "".join(parts)
        return str(content)

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context

        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length

        Returns:
            Generated response string
        """
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template=(
                "You are a helpful document AI assistant. Use the following context "
                "to answer the question accurately and concisely.\n"
                "Cite the original HTML pages using the bracket numbers from the context, "
                "like [1] or [2].\n\n"
                "Context:\n{context}\n\n"
                "Question: {question}\n\n"
                "Answer: Provide a clear and informative answer based on the context above, "
                "with inline citations. If the context doesn't contain enough information "
                "to answer the question, say so."
            ),
        )

        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return self._as_text(response.content)
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting

        Args:
            query: User question
            context: Retrieved context

        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return self._as_text(response.content)
        except Exception as e:
            return f"Error: {str(e)}"

    def answer(self, query: str, top_k: int = 5, simple: bool = False) -> str:
        """Retrieve HTML chunks from the vector store, then generate an LLM answer."""
        retrieved_docs = self.retriever.retrieve(query, top_k=top_k)
        context = self._format_context(retrieved_docs)

        if not context:
            return "No relevant documents were retrieved from the vector store."

        if simple:
            return self.generate_response_simple(query, context)
        return self.generate_response(query, context)


rag_generator = RAGGenerator(rag_retriever)
rag_generator


### Integration: VectorDB context pipeline with LLM output


In [ ]:
class RAGPipeline:
    """Retrieve Unity HTML chunks from ChromaDB and pass them to Gemini as context."""

    def __init__(self, retriever: RAGRetriever, generator: RAGGenerator):
        self.retriever = retriever
        self.generator = generator

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        return_context: bool = False,
    ) -> dict:
        """
        Full RAG path: embed question -> search VectorDB -> format context -> LLM answer.

        Returns answer, sources, confidence (best similarity score), and optionally full context.
        Chunks below min_score are dropped before generation.
        """
        print("=== RAG Pipeline ===")
        print(f"Question: {question}")
        print(f"Top K: {top_k}, Min score: {min_score}")

        retrieved_docs = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score,
        )

        if not retrieved_docs:
            output = {
                "answer": "No relevant context found.",
                "sources": [],
                "citations": [],
                "confidence": 0.0,
            }
            if return_context:
                output["context"] = ""
            print(output["answer"])
            return output

        context = self.generator._format_context(retrieved_docs)
        citations = self.generator.build_citations(retrieved_docs)
        sources = [
            {
                "id": citation["id"],
                "source": citation["source_file"],
                "title": citation["title"],
                "section": citation["section"],
                "url": citation["url"],
                "score": citation["score"],
                "preview": retrieved_docs[citation["id"] - 1]["content"][:300]
                + (
                    "..."
                    if len(retrieved_docs[citation["id"] - 1]["content"]) > 300
                    else ""
                ),
            }
            for citation in citations
        ]
        confidence = max(doc["similarity_score"] for doc in retrieved_docs)

        print(f"Sending {len(retrieved_docs)} retrieved chunks to Gemini...")
        answer = self.generator.generate_response(question, context)

        print("\n--- Answer ---")
        print(answer)
        print(f"\n--- Confidence: {confidence:.3f} ---")
        print("\n--- Citations ---")
        for citation in citations:
            title = citation["title"] or citation["source_file"]
            section = f" — {citation['section']}" if citation["section"] else ""
            print(f"[{citation['id']}] {title}{section}")
            if citation["url"]:
                print(f"    {citation['url']}")

        output = {
            "answer": answer,
            "sources": sources,
            "citations": citations,
            "confidence": confidence,
        }
        if return_context:
            output["context"] = context
        return output


rag_pipeline = RAGPipeline(rag_retriever, rag_generator)
result = rag_pipeline.query(
    "What is a script",
    top_k=5,
    min_score=0.2,
    return_context=True,
)
print("\nConfidence:", result["confidence"])
result["answer"]
